## Ch7-02 — State machine and execution traces

This notebook introduces state usage with transitions (construct 13); after running it you can simulate the toaster's operating cycle for a normal toast run and a cancelled run.


Chapter 4 introduced action flow for the `ApplyHeat` operation. This notebook adds a `state Cycle` that captures the toaster's discrete operating modes — idle, heating, ready, and cancelled — and uses `execute_state` to simulate how events move the system between those modes. See [Ch4-01 action def](../ch04-functional-decomp/01-action-def-ffbd.ipynb) for the action def this state machine complements.


In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
    requirement def HeatingReq {
        subject heater : Heater;
        require constraint { heater.power >= 600.0 }
    }
    requirement heating : HeatingReq;
    part efficient : Heater;
    part weak : Heater { attribute :>> power = 400.0; }
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement {
        attribute resistance : Real default = 12.0;
    }
    part def PowerWire :> HeatingElement {
        attribute gauge : Real default = 14.0;
    }
    part def HeatingAssembly :> HeatingSystem {
        part coil : ResistanceCoil;
        part wire : PowerWire;
    }
    part heatingEvidence {
        assert satisfy heating by efficient;
        assert satisfy heating by weak;
    }
    part def BreadLoader { part bread : Start; }
    part def BreadEjector { part bread : Finish; }
    state Cycle {
        entry; then idle;
        state idle;
        state heating;
        state ready;
        state cancelled;
        transition first idle accept Start then heating;
        transition first heating accept Finish then ready;
        transition first heating accept Cancel then cancelled;
    }

    part def BreadHandling {
        part loader : BreadLoader;
        part ejector : BreadEjector;
        flow loader.bread to ejector.bread;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")


In [ ]:
# A state machine referencing an undefined transition target fails to parse.
bad_source = """
package P {
    item def Go;
    state S {
        entry; then a;
        state a;
        transition first a accept Go then missing_state;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for undefined transition target"
# Expected: diagnostic for 'missing_state' as an unresolved reference
print(f"Negative control ok: bad.ok={bad.ok}")


In [ ]:
# Locate the Cycle state machine by qualified name
cycle = model.find("ToasterDemo::Cycle")
assert cycle is not None, "Cycle not found"
print(f"Cycle: kind={cycle.kind!r}, id={cycle.id!r}")

# Normal run: Start → heating, Finish → ready
normal = model.execute_state(cycle.id, events=["Start", "Finish"])
print(f"Normal trace:  {normal['states_visited']}")
assert normal["states_visited"] == ["idle", "heating", "ready"]

# Cancelled run: Start → heating, Cancel → cancelled
cancelled = model.execute_state(cycle.id, events=["Start", "Cancel"])
print(f"Cancelled trace: {cancelled['states_visited']}")
assert cancelled["states_visited"] == ["idle", "heating", "cancelled"]

conn.close()


The `state Cycle` with four substates and three transitions (A-F) is executed by OpenSysML's `execute_state` (O-S); the states visited — `['idle', 'heating', 'ready']` for a normal run and `['idle', 'heating', 'cancelled']` for a cancel — appear in the result dict (E).


Try the chapter exercise in `exercises/ch07/exercise.ipynb`: add a `state BrewCycle` to the coffee maker model with an `Overheat` transition to a `fault` state, and verify the trace with `execute_state`.
